In [22]:
import pandas as pd
import numpy as np
import datetime
import os

import sklearn
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import  Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import BayesianRidge
from sklearn.utils import estimator_html_repr
from sklearn import set_config

In [23]:
#### import scripts as a module
import sys
sys.path.insert(0, '../scripts')

import helpers

In [25]:
### specify the path to the files
root = '/home/damir/Documents/Projects/sosall/'
out = '/home/damir/Documents/Projects/sosall/results/preprocessing/'

### setup the log file
logger = helpers.setup_logger(path_file = os.path.join(root,'logfile'),
                              path_project = root )

# questionnaire data
data = os.path.join(root, 'results/preprocessing/DataTransformed.txt')
data = pd.read_csv(data,
                   sep = '\t',
                   index_col=0)

# RNAseq dataframe that was reduced to lasso-boruta features
rna = os.path.join(root, 'raw-data/lasso_boruta_rna_data.csv')
rna = pd.read_csv(rna,
                  sep=',',
                  index_col=0)

# file that contains which columns to exclude from the questionnaire data
column_filter = pd.read_csv(os.path.join(root, 'raw-data/column_filter.csv'),
                            header=0,
                            sep = ',')

data.head(100)

,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,childhood_inf_chickenpox,...,fuel_heating_Other,log_blood_counts_wcc,log_blood_counts_hb,log_blood_counts_plts,log_blood_counts_neutrophils,log_blood_counts_monocytes,log_blood_counts_lymphocytes,log_blood_counts_eosinophils,diagnosis_location,RNASeq
PID,,,,,,,,,,,,,,,,,,,,,
CA001LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,0.0,...,0.0,2.380472,2.442347,6.165628,1.229641,-0.162519,1.477049,0.900161,AD_Urban,no
CA002YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,0.0,...,0.0,2.521721,2.433613,6.180224,1.269761,0.292670,1.948763,-0.174353,AD_Urban,yes
CA003US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,0.0,...,0.0,2.078191,2.388763,4.812997,0.837248,-0.713350,1.526056,-0.210721,AD_Urban,no
CA004NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,2.559550,2.572612,5.673667,0.982078,-0.139262,1.968510,0.824175,AD_Urban,no
CA005BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AD_Urban,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CA140EM,AD,Rural,32.0,1.0,1.0,1.0,2.0,1.0,0.0,0.0,...,0.0,2.496506,2.406945,6.066340,1.717395,0.548121,1.523880,-0.634878,AD_Rural,yes
CA141TC,AD,Rural,15.0,2.0,1.0,1.0,3.0,1.0,0.0,0.0,...,0.0,2.628285,2.370244,5.948296,1.451614,0.392042,1.982380,0.086178,AD_Rural,yes
CA142AC,AD,Rural,21.0,2.0,1.0,1.0,3.0,2.0,0.0,0.0,...,0.0,2.409644,2.388763,6.016401,0.824175,-0.116534,1.761300,0.854415,AD_Rural,yes


#### 1. Discarding the variables that do not make sense to impute
- Everything that has more than 20% NAs (medication_oral_which, solidfood_which_first, othernuts_exposure_age_first, othernuts_exposure_regularity, familyhistory_sibling_eczema, familyhistory_sibling_other_allergic_disease,
                                         bsa, medication_antihistamines_last, scorad, eczema_age, eczema_whodiagnosed, peanuts_exposure_age_first, peanuts_exposure_regularity, antibiotic_exposure_age)
- anti_helminthic_exposure_age, anti_helminthic_exposure_last, henegg_exposure_age_first, henegg_exposure_regularity, paracetamol_exposure_age, cowmilkformula_exposure_age_first, household_otherolderchildren,
household_otheryoungerchildren impute with a 0
- peanuts_mother_exposure_avoidance, peanuts_mother_exposure_regularity contains no instead of 0

Update: the file column_filter.csv contains all the values to discard (discussed with Katja)

In [26]:
### drop the columns according to the column filter
drop = column_filter.loc[column_filter.exclude == 1,'variable'].to_list()
data.drop(labels = drop,
          axis = 1,
          inplace = True)
data = data.loc[data.RNASeq == 'yes',:]
### columns to binarize
# sunlight
# vaccination
# pets
# farmanimal
# fuel
# familyhistory
# ever
### extract binary features with yes no
binary_cols = data.loc[:,data.columns.str.contains('pets_|farmanimal_|fuel_|familyhistory_|_ever')].columns
### replace the 1s and 0s with yes and no
#data.loc[:,binary_cols] = data.loc[:,binary_cols].replace(1,'yes')
#data.loc[:,binary_cols] = data.loc[:,binary_cols].replace(0,'no')

# concatenate inputs
data_merged = pd.concat([rna, data],axis=1)
data_merged.head()

In [27]:
# implement imputation and variance thresholding pipline
# define imputation and variance threshold procedures
# imputer for numeric values
imp_br = IterativeImputer(estimator=BayesianRidge(tol=1e-5, n_iter=1000),
                          max_iter=1000,
                          initial_strategy='median')

imp_rf = IterativeImputer(estimator=RandomForestClassifier(n_estimators=100, n_jobs=-1),
                          max_iter=1000,
                          skip_complete=True,
                          n_nearest_features=5)

var = VarianceThreshold(threshold=0.2)


# define separate transformation for numeric and non-numeric columns
num_features = data.select_dtypes(['float64']).columns
cat_features = data.select_dtypes(['object']).columns
rna_features = rna.select_dtypes(['float64']).columns
skip = ['diagnosis_location']

# define separate procedures for numerical and categorical features
num_pipe = Pipeline(steps = [('impute', imp_br),
                             ('var_thr', var)])
cat_pipe = Pipeline(steps=[('impute', imp_rf),
                           ('encode', OneHotEncoder()),
                           ('var_thr',var)])

# make a ColumnTransformer object
ct = ColumnTransformer(transformers = [('drop', 'drop', ['diagnosis',
                                                         'location',
                                                         'diagnosis_location',
                                                         'RNASeq']),
                                       ('num', num_pipe, num_features),
                                       ('cat', cat_pipe, binary_cols),
                                       ('rna','passthrough',rna_features),

                                       ('encode',OneHotEncoder(),['stool_katokatz'])],
                       remainder='passthrough')
pipe = Pipeline(steps = [('ColumnTransformer', ct),
                         ('StandardScaler',StandardScaler())])

In [128]:
# write out the pipeline as html representation
set_config('display_diagram')
with open(os.path.join(root,'preprocessing_pipeline.html'), 'w',encoding='utf-8') as f:
    f.write(estimator_html_repr(pipe))

In [29]:
# transformed questionnaire dataset with implemented imputation and variance threshold
data_transformed = pd.DataFrame(pipe.fit_transform(data_merged),
                                columns = pipe.get_feature_names_out(),
                                index=data_merged.index)

# formatting of the final dataframe for the downstream analysis
data_transformed.columns = data_transformed.columns.str.replace('rna__|simple_imp__|num__|cat__|encode__','')
data_transformed.columns = data_transformed.columns.str.replace('_0.0','no')
data_transformed.columns = data_transformed.columns.str.replace('_1.0','yes')
data_transformed = data_transformed.join(data_merged['diagnosis_location'])
data_transformed = data_transformed.loc[:,~data_transformed.columns.isin(binary_cols)]

# get today's date for the output file
today = datetime.date.today()
data_transformed.to_csv(os.path.join(out, 'IntegratedData_{}.txt'.format(today)),
                        sep='\t')
data_transformed.head(10)

/tmp/ipykernel_1134442/1023477021.py:7: FutureWarning: The default value of regex will change from True to False in a future version.
  data_transformed.columns = data_transformed.columns.str.replace('rna__|simple_imp__|num__|cat__|encode__','')


,enrolement_age,gender,paracetamol_exposure_age,paracetamol_exposure_time,antibiotic_exposure_courses,antibiotic_exposure_last2months,anti_helminthic_exposure_age,anti_helminthic_exposure_last,anti_helminthic_exposure_yearly,delivery_mode,...,ENSGnonononono28449yes__THSD8,ENSGnonononono284755__ALno49557.yes,ENSGnonononono28493yes__ACyesno4389.5,ENSGnonononono284989__AL45yesno62.3,ENSGnonononono285283__ALno35no78.4,ENSGnonononono285839__AL445685.3,stool_katokatz_No stool,stool_katokatz_negative,stool_katokatz_positive,diagnosis_location
CA002YF,-1.158667,-0.849728,1.311994,-0.460464,-0.161102,-0.785054,-0.018184,-1.318086,-0.108225,-0.651941,...,-1.814697,0.261213,0.175599,1.525850,0.698366,0.574049,1.709109,-1.520234,-0.222027,AD_Urban
CA009ST,0.256426,-0.849728,0.357508,-2.000404,-1.476194,1.299754,-0.250893,1.053388,0.701397,-0.651941,...,-0.676496,-1.213820,0.868272,0.358090,1.101734,2.016199,-0.585100,0.657794,-0.222027,AD_Urban
CA010EB,0.256426,1.183352,0.048810,-0.005443,0.161830,-0.027375,0.004207,0.439997,0.049551,-0.099765,...,0.753810,0.816059,0.039795,0.238299,1.279066,0.411856,-0.585100,0.657794,-0.222027,AD_Urban
CA011LQ,1.246992,-0.849728,0.072940,-0.460464,1.153990,1.299754,-0.250893,1.224794,0.701397,1.543220,...,-0.648041,-1.558708,-0.148462,0.856801,-0.950623,-0.870003,-0.585100,0.657794,-0.222027,AD_Urban
CA014LB,-0.875648,1.183352,0.485958,2.619416,1.153990,1.299754,-0.250893,-0.146455,0.701397,-0.651941,...,-1.227935,0.908408,-0.546354,-0.009247,-0.867322,1.274924,-0.585100,0.657794,-0.222027,AD_Urban
CA015AM,-0.026592,-0.849728,-1.166114,2.619416,1.153990,1.299754,2.945652,0.196357,-1.445409,1.543220,...,-1.320169,0.320843,-0.562341,-0.977353,0.106218,0.269506,-0.585100,0.657794,-0.222027,AD_Urban
CA020AZ,0.114917,-0.849728,-0.036670,-2.000404,-1.476194,-0.785054,-0.250893,-0.832079,-1.445409,-0.651941,...,-1.310356,0.110921,0.001497,-0.977353,-0.388984,-0.870003,-0.585100,0.657794,-0.222027,AD_Urban
CA021IM,-1.441686,-0.849728,-0.753096,1.079476,-1.476194,-0.785054,-0.025905,-1.780393,-1.445409,1.543220,...,-0.595056,0.609843,0.798933,0.034182,-1.325730,-0.870003,1.709109,-1.520234,-0.222027,AD_Urban
CA022MS,1.530010,1.183352,0.072940,-0.460464,0.069706,-0.785054,0.814622,1.224794,0.701397,-0.651941,...,-1.814697,-0.109222,1.534718,0.453996,-0.693056,0.655680,1.709109,-1.520234,-0.222027,AD_Urban
CA023EJ,0.114917,1.183352,-0.340078,1.079476,-0.161102,-0.785054,2.945652,0.196357,-1.445409,-0.651941,...,-0.824659,0.848336,1.640344,1.200735,0.505140,0.849462,-0.585100,0.657794,-0.222027,AD_Urban
